In [35]:
%matplotlib qt
import hyperspy.api as hs
import pyxem as pxm

In [36]:
import gc as gc

In [37]:
gc.collect()

40141

#### Load data

In [38]:
s = hs.load(r'C:\Users\annam\Downloads\comp files\yr4\cotic4f_Cropped150x150_Startx90y65_8bit (1).hspy')

In [39]:
s.data.shape

(151, 151, 257, 257)

#### Remove camera pixels that are 'hot' (always a bright intensity - not working pixels)

In [40]:
meanDP = s.mean((0,1))

In [41]:
meanDP.plot(vmin=0.,vmax=64)

In [42]:
hot_pix = meanDP.find_hot_pixels(threshold_multiplier=30,lazy_result=False)

c:\Users\annam\HyperSpy-bundle\Lib\site-packages\pyxem\signals\diffraction2d.py:516: VisibleDeprecationWarning: Argument `lazy_result` is deprecated and will be removed in version 1.0.0. To avoid this warning, please do not use `lazy_result`. Use `lazy_output` instead. See the documentation of `find_hot_pixels()` for more details.
  name="mask_array", since="0.15.0", removal="1.0.0", alternative="mask"


  0%|          | 0/2 [00:00<?, ?it/s]

In [43]:
s.data[:,:,hot_pix]=0

#### Centre the direct beam (centre of the pattern 'wobbles' due to imperfect tilt/shift of the beam)

In [44]:
s.center_direct_beam(method='center_of_mass')

  0%|          | 0/76 [00:00<?, ?it/s]

In [ ]:
s.center_direct_beam(method='center_of_mass',mask=(128,128,10)) #remove this line ? this section?

  0%|          | 0/76 [00:00<?, ?it/s]

In [46]:
s.plot(vmin=0.,vmax=50.)

In [47]:
s.save(r'C:\Users\annam\Downloads\comp files\yr4\cotic4f_Cropped150x150_Startx90y65_8bit (2)HotPixelsRemoved_Centred.hspy')

#### Create an overview image

In [48]:
im = s.sum((2,3))
im.plot()

In [49]:
im.data

array([[21600, 20877, 21224, ..., 21375, 19999, 20122],
       [20304, 21607, 21940, ..., 20119, 20015, 19409],
       [21453, 21668, 22237, ..., 20508, 20069, 20127],
       ...,
       [24812, 24345, 26003, ..., 22645, 23164, 23962],
       [23139, 25978, 24525, ..., 24060, 22607, 23389],
       [24048, 23982, 25604, ..., 23306, 22906, 24002]], dtype=uint32)

In [50]:
im.change_dtype('float64')
im.data -= im.data.min()
im.data /= im.data.max()
im.data *= 255.
im.change_dtype('uint8')
im.as_signal2D((0,1)).save(r'C:\Users\annam\Downloads\comp files\yr4\cotic4f_8bit.tiff')

In [51]:
im.plot()

### Create average diffraction patterns from regions of interest #remove this section?

In [52]:
dp_Area1 = s.inav[114.:239.,65.:112.].mean((0,1))
dp_Area1.plot(vmin=0.,vmax=10)

In [53]:
dp_Area2 = s.inav[90.:145.,168.:215.].mean((0,1))
dp_Area2.plot(vmin=0.,vmax=10)

In [54]:
dp_Area3 = s.inav[199.:239.,166.:215.].mean((0,1))
dp_Area3.plot(vmin=0.,vmax=30)

#### Plot a Virtual dark field (image created from intensity from a specific area of the diffraction data)

In [55]:
vap = hs.roi.CircleROI(cx=128.,cy=128.,r=3.)

In [56]:
s.plot_integrated_intensity(vap)

In [57]:
# Check the pixel scale of the diffraction axes (usually axes 2 and 3)
print(f"Original Scale: {s.axes_manager[2].scale} A^-1 per pixel")

# Check what the scale will be IF you rebin by 4 (to get to 64x64)
rebin_factor = 4
new_scale = s.axes_manager[2].scale * rebin_factor
print(f"Rebinned Scale (64x64): {new_scale} A^-1 per pixel")

Original Scale: 1.0 A^-1 per pixel
Rebinned Scale (64x64): 4.0 A^-1 per pixel


In [30]:
# Look at the central beam of your centered data
s.inav[0,0].plot(vmin=0, vmax=100)
# Zoom into the center manually in the plot window

In [31]:
import matplotlib.pyplot as plt
import numpy as np
import gc

# 1. Close active plots to prevent coordinate/ROI errors
plt.close('all')
gc.collect()

# 2. Change data to float32 to save 50% RAM
s.change_dtype('float32')

# 3. Set the physical calibration (0.8 mrad @ 300kV)
s.axes_manager[2].scale = 0.008
s.axes_manager[3].scale = 0.008
s.axes_manager[2].units = '1/A'
s.axes_manager[3].units = '1/A'

# 4. Rebin to 64x64 (Detects your scan size—150 or 151—automatically)
# This forces the signal axes to 64x64 and updates scale to 0.032
ny, nx = s.data.shape[0], s.data.shape[1]
s_64 = s.rebin(new_shape=(ny, nx, 64, 64))

print(f"Success! Data shape: {s_64.data.shape}")
print(f"Final Scale: {s_64.axes_manager[2].scale:.3f} A^-1/px")

Success! Data shape: (151, 151, 64, 64)
Final Scale: 0.032 A^-1/px


In [32]:
import tensorflow as tf

# 1. Load the Denoising Autoencoder
# Change filename to 'denoiser_v2.h5' if you haven't renamed your file yet
dae_model = tf.keras.models.load_model('denoiser_v2.keras')

# 2. Get dimensions automatically
nav_y, nav_x, sig_y, sig_x = s_64.data.shape

# 3. Reshape and Normalize for the AI
# We use -1 so it automatically counts your 150x150 (or 151x151) patterns
cotic_reshaped = s_64.data.reshape(-1, sig_y, sig_x, 1)

cotic_min = cotic_reshaped.min(axis=(1, 2, 3), keepdims=True)
cotic_max = cotic_reshaped.max(axis=(1, 2, 3), keepdims=True)
cotic_norm = (cotic_reshaped - cotic_min) / (cotic_max - cotic_min + 1e-12)

# 4. Run Denoising
print(f"Denoising {nav_y * nav_x} patterns... please wait.")
denoised_flat = dae_model.predict(cotic_norm, batch_size=32)

# 5. Reshape back to the 4D scan grid
denoised_data = denoised_flat.reshape(nav_y, nav_x, sig_y, sig_x)

print("Denoising complete! Clean patterns are in 'denoised_data'.")

Denoising 22801 patterns... please wait.
713/713 ━━━━━━━━━━━━━━━━━━━━ 21s 29ms/step
Denoising complete! Clean patterns are in 'denoised_data'.


In [33]:
# Select a test pixel (the middle of your scan)
ry, rx = nav_y // 2, nav_x // 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Original Noisy Data
ax1.imshow(s_64.data[ry, rx], cmap='magma')
ax1.set_title(f"Original Noisy COTIC\n(Pixel: {ry}, {rx})")

# DAE Denoised Data
ax2.imshow(denoised_data[ry, rx], cmap='magma')
ax2.set_title("DAE Denoised Output\n(Physical Scale: 0.032 A^-1/px)")

plt.tight_layout()
plt.show()

In [34]:
# Change the test pixel to 10, 10
ry, rx = 10, 10

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Original Noisy Data at 10, 10
ax1.imshow(s_64.data[ry, rx], cmap='magma')
ax1.set_title(f"Original Noisy COTIC\n(Pixel: {ry}, {rx})")

# DAE Denoised Data at 10, 10
ax2.imshow(denoised_data[ry, rx], cmap='magma')
ax2.set_title("DAE Denoised Output\n(Physical Scale: 0.032 A^-1/px)")

plt.tight_layout()
plt.show()